# Bird Identification using CV - Part 6: Species Categorization

**ITAI 1378  |  Midterm  |  2026**

**Group 8**

**Author:** Stuart Fairchild | Kalen Foster | Ranveer Chand

---

Train bird classification model. Training on species normally found at the feeder that are in the dataset.



In [1]:
%pip install -q ultralytics

from ultralytics import YOLO, SAM
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

print("Setup complete. Ready to detect and segment.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete. Ready to detect and segment.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Create data.yaml file

In [3]:
import yaml
import os

# Define the base dataset path to the common parent directory for all species
dataset_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/'

# Define paths relative to the dataset_path for images and labels
# These should point to the consolidated train/val folders containing all species data
train_dir = os.path.join(dataset_path, 'train')
val_dir = os.path.join(dataset_path, 'val')

# Check if the directories exist (optional, but good practice)
if not os.path.exists(train_dir):
    print(f"Warning: Training directory not found at {train_dir}. Please ensure data is consolidated.")
if not os.path.exists(val_dir):
    print(f"Warning: Validation directory not found at {val_dir}. Please ensure data is consolidated.")

# Define the content for data.yaml with updated class information
data_yaml_content = {
    'path': dataset_path, # Root directory where 'train' and 'val' folders are
    'train': 'train/images', # Relative path to training images from 'path'
    'val': 'val/images',     # Relative path to validation images from 'path'
    'train_labels': 'train/labels', # Relative path to training labels from 'path'
    'val_labels': 'val/labels',     # Relative path to validation labels from 'path'
    'nc': 7,                 # Number of classes (American Robin, Blue Jay, Downy Woodpecker, House Finch, Mourning Dove, Northern Cardinal, Red-bellied Woodpecker)
    'names': ['American Robin', 'Blue Jay', 'Downy Woodpecker', 'House Finch', 'Mourning Dove', 'Northern Cardinal', 'Red-bellied Woodpecker'] # Class names
    # not in dataset but seen: White-winged dove, european starling, brown-headed nuthatch
}

# Define the full path for the data.yaml file
data_yaml_file_path = os.path.join(dataset_path, 'data.yaml')

# Write the dictionary to a YAML file
with open(data_yaml_file_path, 'w') as file:
    yaml.dump(data_yaml_content, file, default_flow_style=False)

print(f"data.yaml created successfully at: {data_yaml_file_path}")
print("Content of data.yaml:")
print(yaml.dump(data_yaml_content, default_flow_style=False))

data.yaml created successfully at: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/data.yaml
Content of data.yaml:
names:
- American Robin
- Blue Jay
- Downy Woodpecker
- House Finch
- Mourning Dove
- Northern Cardinal
- Red-bellied Woodpecker
nc: 7
path: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/
train: train/images
train_labels: train/labels
val: val/images
val_labels: val/labels



## Correct Class IDs in Label Files

Since the data files were prepared separately and the `class_id` might have defaulted to `0`, this script will iterate through the label files in the `train/labels` and `val/labels` directories.

It will infer the correct species from the filename (e.g., `Blue_Jay_...` for 'Blue Jay') and update the `class_id` in each line of the label file to match the `data.yaml`'s class mapping.

In [5]:
import os
import yaml

# Define the base dataset path where species subfolders are located
base_dataset_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/'

# --- Dynamically get class names and build species_to_class_id from data.yaml ---
# Assuming data.yaml is located at the base_dataset_path
data_yaml_file_path = os.path.join(base_dataset_path, 'data.yaml')
species_to_class_id = {}

if os.path.exists(data_yaml_file_path):
    try:
        with open(data_yaml_file_path, 'r') as f:
            data_config = yaml.safe_load(f)
        class_names_from_yaml = data_config.get('names', [])
        # Create mapping: folder_name (with space replaced by underscore) -> class_id
        species_to_class_id = {name.replace(' ', '_'): i for i, name in enumerate(class_names_from_yaml)}
        print("Dynamically loaded species to class ID mapping from data.yaml:")
        print(species_to_class_id)
    except Exception as e:
        print(f"Error loading or parsing data.yaml: {e}. Using hardcoded mapping.")
else:
    print(f"Warning: data.yaml not found at {data_yaml_file_path}. Using hardcoded species_to_class_id.")

# Fallback to hardcoded mapping if data.yaml not found or parsing failed
# if not species_to_class_id:
#     species_to_class_id = {
#         'Blue_Jay': 0,
#         'Downy_Woodpecker': 1,
#         'Mourning_Dove': 2,
#         'Red-bellied_Woodpecker': 3
#     }
#     print("Using hardcoded species to class ID mapping as a fallback.")
#     print(species_to_class_id)

# Dynamically find species subfolders to process their label files
label_dirs_to_process = []
for item_name in os.listdir(base_dataset_path):
    item_path = os.path.join(base_dataset_path, item_name)
    # Check if it's a directory and its name (with spaces replaced) is in our species mapping
    # Exclude 'train' and 'val' if they exist as consolidated folders
    if os.path.isdir(item_path) and item_name not in ['train', 'val'] and item_name.replace(' ', '_') in species_to_class_id:
        label_dirs_to_process.append(item_path)

if not label_dirs_to_process:
    print(f"No species subfolders found in {base_dataset_path} that match the defined classes. Please check your folder names and data.yaml.")
    print("This script assumes the labels are in these subfolders *before* a train/val split or on the original dataset.")

print("Starting correction of class IDs in label files...")

processed_files_count = 0
for label_dir in label_dirs_to_process:
    # Extract species name from the folder name (e.g., 'Blue Jay' from '.../preprocessed/Blue Jay')
    species_folder_name = os.path.basename(label_dir)
    species_key_for_lookup = species_folder_name.replace(' ', '_')

    if species_key_for_lookup not in species_to_class_id:
        print(f"Warning: Folder name '{species_folder_name}' does not match any known species in the mapping. Skipping directory {label_dir}.")
        continue

    correct_class_id = species_to_class_id[species_key_for_lookup]
    print(f"Processing labels for species '{species_folder_name}' (Class ID: {correct_class_id}) in directory: {label_dir}")

    if not os.path.exists(label_dir):
        print(f"Warning: Label directory not found at {label_dir}. Skipping.")
        continue

    for label_filename in os.listdir(label_dir):
        if label_filename.endswith('.txt'):
            label_filepath = os.path.join(label_dir, label_filename)

            modified_lines = []
            try:
                with open(label_filepath, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            # Change the first element (class_id) to the correct one
                            parts[0] = str(correct_class_id)
                            modified_lines.append(' '.join(parts))
                        else:
                            modified_lines.append('') # Keep empty lines if any

                # Write the modified content back to the file
                with open(label_filepath, 'w') as f:
                    for line in modified_lines:
                        f.write(line + '\n')
                processed_files_count += 1
            except Exception as e:
                print(f"Error processing {label_filepath}: {e}")

print(f"Finished correcting class IDs. Processed {processed_files_count} label files.")
print("If the train/val split was already performed, you may need to re-run it after this correction.")
print("Please verify a few label files manually to ensure correct class IDs.")

Dynamically loaded species to class ID mapping from data.yaml:
{'American_Robin': 0, 'Blue_Jay': 1, 'Downy_Woodpecker': 2, 'House_Finch': 3, 'Mourning_Dove': 4, 'Northern_Cardinal': 5, 'Red-bellied_Woodpecker': 6}
Starting correction of class IDs in label files...
Processing labels for species 'Northern Cardinal' (Class ID: 5) in directory: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/Northern Cardinal
Processing labels for species 'Blue Jay' (Class ID: 1) in directory: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/Blue Jay
Processing labels for species 'Downy Woodpecker' (Class ID: 2) in directory: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/Downy Woodpecker
Processing labels for species 'Mourning Dove' (Class ID: 4) in directory: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/Mourning Dove
Processing labels for species 'Red-

## Generate Train/Validation Split

This section will split your dataset into training and validation sets. It assumes that:
1.  Your images (e.g., `.jpg`, `.png`) are located directly within the `dataset_path`.
2.  For each image, there is a corresponding label file (`.txt`) with the same base name in the same `dataset_path` or a `labels` subdirectory if you have them organized that way.

It will then create `train/images`, `train/labels`, `val/images`, `val/labels` subdirectories within your `dataset_path` and move the files accordingly. If you have images without corresponding label files, they will be skipped, but YOLO training will require all images to have a label file (even if empty).


In [6]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

# Define the base dataset path to the common parent directory for all species
# This is where the species subfolders are located (e.g., 'Blue Jay', 'Downy Woodpecker').
base_dataset_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/'

# Define the consolidated directories for the split dataset, directly under base_dataset_path
train_img_dir = os.path.join(base_dataset_path, 'train', 'images')
train_lbl_dir = os.path.join(base_dataset_path, 'train', 'labels')
val_img_dir = os.path.join(base_dataset_path, 'val', 'images')
val_lbl_dir = os.path.join(base_dataset_path, 'val', 'labels')

# Create target consolidated directories if they don't exist
for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

print(f"Target consolidated directories created:\n{train_img_dir}\n{train_lbl_dir}\n{val_img_dir}\n{val_lbl_dir}")

all_data_pairs = []
found_species_folders = []

# Iterate through subdirectories (species folders) within the base_dataset_path
for item_name in os.listdir(base_dataset_path):
    item_path = os.path.join(base_dataset_path, item_name)

    # Ensure it's a directory and not one of the new train/val dirs or the data.yaml
    if os.path.isdir(item_path) and item_name not in ['train', 'val']:
        species_folder_path = item_path
        species_folder_name = item_name

        found_species_folders.append(species_folder_name)
        print(f"Processing species folder: {species_folder_name}")

        # List all image files in the current species folder
        species_image_files = []
        for f in os.listdir(species_folder_path):
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                species_image_files.append(os.path.join(species_folder_path, f))

        if not species_image_files:
            print(f"No image files found in {species_folder_path}. Skipping this species folder.")
            continue # Move to the next species folder

        # Prepare a list of (image_path, label_path) tuples for the current species
        for img_path in species_image_files:
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(species_folder_path, base_name + '.txt')

            if os.path.exists(label_path):
                all_data_pairs.append((img_path, label_path))
            else:
                print(f"Warning: No label file found for {os.path.basename(img_path)} in {species_folder_name}. Skipping this image.")

if not found_species_folders:
    print(f"No species subfolders found in {base_dataset_path}. Please ensure your dataset structure is correct (e.g., '{base_dataset_path}Blue Jay/').")
elif not all_data_pairs:
    print("No image-label pairs found across all species folders. Please ensure images have corresponding .txt label files in their respective species subfolders.")
else:
    print(f"Found {len(all_data_pairs)} total image-label pairs across all species for splitting.")

    # Split data into training and validation sets
    train_data, val_data = train_test_split(all_data_pairs, test_size=0.2, random_state=42) # 80% train, 20% validation

    print(f"Moving {len(train_data)} pairs to training set...")
    # Move files for training set
    for img_src, lbl_src in train_data:
        img_name = os.path.basename(img_src)
        lbl_name = os.path.basename(lbl_src)
        shutil.move(img_src, os.path.join(train_img_dir, img_name))
        shutil.move(lbl_src, os.path.join(train_lbl_dir, lbl_name))

    print(f"Moving {len(val_data)} pairs to validation set...")
    # Move files for validation set
    for img_src, lbl_src in val_data:
        img_name = os.path.basename(img_src)
        lbl_name = os.path.basename(lbl_src)
        shutil.move(img_src, os.path.join(val_img_dir, img_name))
        shutil.move(lbl_src, os.path.join(val_lbl_dir, lbl_name))

    print("Dataset split and moved successfully to consolidated train/val folders!")
    print(f"Total Training images: {len(train_data)}, Total Validation images: {len(val_data)}")

Target consolidated directories created:
/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/train/images
/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/train/labels
/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/val/images
/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/val/labels
Processing species folder: Northern Cardinal
Processing species folder: Blue Jay
Processing species folder: Downy Woodpecker
Processing species folder: Mourning Dove
Processing species folder: Red-bellied Woodpecker
Processing species folder: American Robin
Processing species folder: House Finch
Found 14840 total image-label pairs across all species for splitting.
Moving 11872 pairs to training set...
Moving 2968 pairs to validation set...
Dataset split and moved successfully to consolidated train/val folders!
Total Training images: 11872, Total Validation imag

## Train YOLO model for multi bird detection

In [7]:
import os
from ultralytics import YOLO

# Define the base dataset path to the common parent directory for all species
dataset_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/'

# Initialize a YOLO model for training from scratch
model_trainer = YOLO('yolo11l.pt') # Load a base model for training

# The data.yaml file is now expected at the root of the dataset_path
data_yaml_path = os.path.join(dataset_path, 'data.yaml')

print(f"Attempting to train YOLO model using dataset at: {dataset_path}")
print(f"Expecting data.yaml at: {data_yaml_path}")

results = model_trainer.train(
    data=data_yaml_path, # Path to your dataset configuration file
    epochs=50,             # Number of training epochs
    imgsz=640,             # Image size for training
    batch=16,              # Batch size
    name='yolo_multi_bird_detector' # Name for the training run (updated for multi-species)
)

print("Model training complete.")

Attempting to train YOLO model using dataset at: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/
Expecting data.yaml at: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/data.yaml
Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing

## Export the model

In [12]:
import os
import shutil

# Define the path to the runs directory
runs_dir = '/content/runs'
zip_file_name = 'runs.zip'
zip_file_path = os.path.join('/content/', zip_file_name)

# Create a zip archive of the runs directory
shutil.make_archive(os.path.splitext(zip_file_path)[0], 'zip', runs_dir)

print(f"Successfully created {zip_file_name} at {zip_file_path}")

Successfully created runs.zip at /content/runs.zip
